# Pretrain simclr on HyperKvasir unlabeled

ViT-S/16 @ 224, global batch 512, 100 epochs. Estimated **~15.8 GPU-hours** (~3 session(s) at the 7.5h guard).

**This notebook is resumable.** It stops cleanly before the session cap, saves full training state (model, EMA target, optimizer, scaler, schedule position) to a Kaggle Dataset, and picks up exactly where it left off next run. Just *Save & Run All* again until it prints `run complete`.

Requires **GPU T4 x2** and Internet ON, plus `KAGGLE_USERNAME`/`KAGGLE_KEY` under Add-ons → Secrets for cross-session checkpointing.

In [ ]:
REPO_URL = "https://github.com/morsalin101/jepa-thesis.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/jepa-thesis"


In [ ]:
import os, subprocess, sys

def sh(cmd, check=True):
    """Run a shell command, streaming its output live.

    Streaming rather than capture_output matters here: the corpus resize runs
    for 20-40 minutes and prints a progress line with an ETA every 256 images.
    Buffering that until the process exits makes a long job indistinguishable
    from a hung one.
    """
    print('$', cmd, flush=True)
    # PYTHONUNBUFFERED: a child process writing to a pipe switches from line
    # buffering to 4 KB block buffering, so progress lines would still arrive
    # in bursts (or not at all until exit) even though we stream them here.
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    lines = []
    for line in p.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code_ = p.wait()
    if check and code_ != 0:
        # Include the tail of the output in the exception. Otherwise the
        # traceback shows only this wrapper and the real error is buried
        # further up the cell, which is easy to miss and impossible to
        # copy/paste usefully.
        tail = ''.join(lines[-25:]).rstrip()
        raise RuntimeError(
            f'command failed (exit {code_}): {cmd}\n\n--- last output ---\n{tail}')
    return subprocess.CompletedProcess(cmd, code_, ''.join(lines), '')

if subprocess.run(f'git ls-remote {REPO_URL}', shell=True,
                  capture_output=True).returncode != 0:
    raise RuntimeError('Cannot reach GitHub — turn Internet ON in the session options.')

if os.path.exists(WORKDIR):
    sh(f'cd {WORKDIR} && git fetch origin && git reset --hard origin/{BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {REPO_URL} {WORKDIR}')
sh(f'cd {WORKDIR} && git log -1 --oneline')
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)


In [ ]:
sh('pip install -q -r requirements.txt')


In [ ]:
# Report the accelerator and the precision that follows from it.
# T4 (sm_75) has fp16 tensor cores but NO bf16 hardware; P100 (sm_60) has
# neither and runs ~2x slower. The code adapts either way — this cell is
# here so you know what you were given before spending 8 hours on it.
import torch
from src.config import amp_config
print('CUDA devices:', torch.cuda.device_count())
amp = amp_config()
print(amp)
if torch.cuda.device_count() < 2:
    print('\n*** Only one GPU. I-JEPA and MAE will still run correctly, but\n'
          '    SimCLR/MoCo v3 need 2 GPUs to preserve global_batch=512.\n'
          '    Set Session options -> Accelerator -> GPU T4 x2 and re-run. ***')


In [ ]:
# ---- run configuration -------------------------------------------------
EPOCHS = 100      # 100 = paper-faithful. 30 is a reasonable budget-cut.
MAX_IMAGES = 0    # 0 = whole corpus. Set e.g. 3000 for a quick pipeline check.
GUARD_HOURS = 7.5 # stop cleanly before Kaggle's session cap
# ------------------------------------------------------------------------

# Cost at the full 99,417-image corpus, T4 x2:
#   simclr: ~9.5 min/epoch
# Whatever you pick, use the SAME budget for all four methods — that
# equality is what makes the comparison a comparison.
print(f'{EPOCHS} epochs, max_images={MAX_IMAGES or "all"}')


In [ ]:
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = s.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = s.get_secret('KAGGLE_KEY')


In [ ]:
mi = f'--max-images {MAX_IMAGES}' if MAX_IMAGES else ''
sh(f'python -m src.engine.pretrain --method simclr --epochs {EPOCHS} {mi} --ckpt-slug morsalin101/jepa-thesis-ckpt --guard-hours {GUARD_HOURS}', check=False)


In [ ]:
# Loss curve and per-epoch table, read from the metrics written during
# training. Works whether the run finished or the guard stopped it early.
import json, glob
import matplotlib.pyplot as plt

recs = [json.loads(l) for l in
        open('/kaggle/working/ckpt/metrics.jsonl')] \
       if glob.glob('/kaggle/working/ckpt/metrics.jsonl') else []
recs = [r for r in recs if r['method'] == 'simclr']

if not recs:
    print('no metrics yet — training has not completed an epoch')
else:
    print(f"corpus: {recs[-1].get('corpus_size', '?')} images | "
          f"{recs[-1]['epoch']} epochs done | "
          f"{sum(r.get('epoch_time_s', 0) for r in recs)/3600:.2f} GPU-h")
    print(f"loss: {recs[0]['loss']:.4f} -> {recs[-1]['loss']:.4f}")
    print()
    print(f"{'epoch':>6} {'loss':>10} {'lr':>10} {'wd':>8} {'min/ep':>8}")
    for r in recs[-15:]:
        print(f"{r['epoch']:6d} {r['loss']:10.4f} {r['lr']:10.2e} "
              f"{r['wd']:8.3f} {r.get('epoch_time_s',0)/60:8.1f}")

    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    ax[0].plot([r['epoch'] for r in recs], [r['loss'] for r in recs])
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].set_title('training loss')
    ax[0].grid(alpha=.3)
    ax[1].plot([r['epoch'] for r in recs], [r['lr'] for r in recs])
    ax[1].set_xlabel('epoch'); ax[1].set_ylabel('lr'); ax[1].set_title('learning rate')
    ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()


In [ ]:
# If the cell above printed 'N epochs remaining', the session guard stopped it
# cleanly — just Save & Run All again to continue. If it printed 'run complete',
# the exported encoder is in /kaggle/working/weights/ and the next cell ships it.
import pathlib
w = pathlib.Path('/kaggle/working/weights')
files = sorted(w.glob('*.pt')) if w.is_dir() else []
for f in files:
    print(f'{f.name}  {f.stat().st_size/1e6:.0f} MB')
if not files:
    print('no encoder yet — the run has not finished; Save & Run All again')


In [ ]:
# Publish the finished encoder (~88 MB) to the shared weights dataset that the
# segmentation notebook mounts. Safe to re-run; a no-op until the run completes.
import glob, json, pathlib, shutil, subprocess

SLUG = 'morsalin101/jepa-thesis-weights'
found = glob.glob('/kaggle/working/weights/*.pt')
if not found:
    print('nothing to publish yet — pretraining has not finished')
else:
    # Stage one level down so --dir-mode zip uploads a single archive
    # instead of one HTTP request per file.
    stage = pathlib.Path('/kaggle/working/weights_upload')
    inner = stage / 'weights'
    inner.mkdir(parents=True, exist_ok=True)

    # Carry over encoders already in the dataset: `datasets version` REPLACES
    # the whole contents, and --delete-old-versions makes that irreversible.
    # Search recursively — Kaggle now nests mounts as
    # /kaggle/input/datasets/<owner>/<slug>/, so a fixed path finds nothing
    # and the previous methods' encoders would be silently destroyed.
    carried = [p for p in glob.glob('/kaggle/input/**/*.pt', recursive=True)
               if 'jepa-thesis-weights' in p]
    for p in carried:
        shutil.copy(p, inner)
    print(f'carried over {len(carried)} existing encoder(s):',
          sorted(os.path.basename(p) for p in carried) or 'none')
    for p in found:
        shutil.copy(p, inner)
    (stage / 'dataset-metadata.json').write_text(json.dumps(
        {'title': 'jepa-thesis-weights', 'id': SLUG,
         'licenses': [{'name': 'CC0-1.0'}]}, indent=2))
    # `datasets status` exits 0 even on a 403/404, so read stdout not the code.
    probe = subprocess.run(['kaggle','datasets','status',SLUG],
                           capture_output=True, text=True)
    pout = (probe.stdout or '') + (probe.stderr or '')
    exists = bool(pout.strip()) and 'error' not in pout.lower()
    # Private is the default; there is no --private flag.
    cmd = (['kaggle','datasets','version','-p',str(stage),'-m',
            'add simclr','--dir-mode','zip','--delete-old-versions']
           if exists else
           ['kaggle','datasets','create','-p',str(stage),'--dir-mode','zip'])
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout or '') + (r.stderr or ''))
    print('published:', sorted(p.name for p in inner.glob('*.pt')))
